# Lecture 5 · Notebook 2 — Scaling laws: train tiny GPTs and fit your own power law
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture05/lecture05_2_scaling_laws.ipynb)

**Companion to [Chapter 1 (slide 11)](https://nlp-course-cuhksz.github.io/lecture05.html#p1-s11) and [Chapter 7 (slide 102)](https://nlp-course-cuhksz.github.io/lecture05.html#p7-s102) of the video lecture.**
Kaplan et al. (2020) found that test loss falls as a **power law** in parameters, data and compute: a straight line on a log-log plot.
Hoffmann et al. (2022, *Chinchilla*) then asked how to split a fixed compute budget between model size and training tokens. You will:

1. train six tiny character-level GPTs (about 3K to 800K non-embedding parameters) on Shakespeare, all on the same data,
2. fit `loss = a · N^(-alpha) + c` and extrapolate to a bigger model,
3. plot loss against training compute and see the *compute-efficient frontier*, and
4. use the published Chinchilla fit to find the compute-optimal model size, and see why GPT-3 was "too big for its data".

Runs on a CPU runtime in about ten minutes; with a free T4 GPU (*Runtime → Change runtime type*) it takes a couple of minutes. Our models are a million times smaller than real LLMs, so expect the
*shape* of the curves, not the exact exponents of the papers.

In [ ]:
import math, time, requests, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import curve_fit

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
text = requests.get("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", timeout=60).text
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print(f"{len(text):,} characters, vocabulary {len(chars)}, device {device}")

## 1 · A minimal GPT
Token and position embeddings, a stack of pre-norm Transformer blocks with a causal mask (Lecture 4), and a linear head that predicts the
next character. Following Kaplan et al., we count **non-embedding parameters** N.

In [ ]:
CTX = 64

class TinyGPT(nn.Module):
    def __init__(self, vocab, d, n_layer, n_head):
        super().__init__()
        self.tok = nn.Embedding(vocab, d); self.pos = nn.Embedding(CTX, d)
        layer = nn.TransformerEncoderLayer(d, n_head, 4 * d, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
        self.blocks = nn.TransformerEncoder(layer, n_layer, enable_nested_tensor=False)
        self.ln = nn.LayerNorm(d); self.head = nn.Linear(d, vocab)
        self.register_buffer("mask", nn.Transformer.generate_square_subsequent_mask(CTX))

    def forward(self, x):
        T = x.shape[1]
        h = self.tok(x) + self.pos(torch.arange(T, device=x.device))
        h = self.blocks(h, mask=self.mask[:T, :T], is_causal=True)
        return self.head(self.ln(h))

def non_embedding_params(m):
    return sum(p.numel() for name, p in m.named_parameters() if not name.startswith(("tok", "pos", "head")))

def batch(split, bs=32):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - CTX - 1, (bs,))
    x = torch.stack([d[i:i + CTX] for i in ix]); y = torch.stack([d[i + 1:i + CTX + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def val_loss(m, iters=20):
    m.eval(); g = torch.Generator().manual_seed(1); losses = []
    for _ in range(iters):
        ix = torch.randint(len(val_data) - CTX - 1, (64,), generator=g)
        x = torch.stack([val_data[i:i + CTX] for i in ix]).to(device); y = torch.stack([val_data[i + 1:i + CTX + 1] for i in ix]).to(device)
        losses.append(F.cross_entropy(m(x).flatten(0, 1), y.flatten()).item())
    m.train(); return float(np.mean(losses))

## 2 · Train six model sizes on the same data
Every model sees the same number of tokens (`STEPS × 32 × 64`, about 2.5M characters, under three epochs). We record the validation loss along the way.

In [ ]:
configs = [(1, 16, 2), (1, 32, 2), (2, 48, 4), (2, 64, 4), (3, 96, 4), (4, 128, 4)]   # (layers, width, heads)
STEPS, EVAL_EVERY, BS = 1200, 100, 32
runs = []
for n_layer, d, n_head in configs:
    torch.manual_seed(0)
    m = TinyGPT(len(chars), d, n_layer, n_head).to(device)
    N = non_embedding_params(m)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3 if d <= 64 else 1.5e-3, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=opt.param_groups[0]["lr"], total_steps=STEPS, pct_start=0.05)
    t0, curve = time.time(), []
    for step in range(1, STEPS + 1):
        x, y = batch("train", BS)
        loss = F.cross_entropy(m(x).flatten(0, 1), y.flatten())
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if step % EVAL_EVERY == 0:
            curve.append((step * BS * CTX, val_loss(m)))
    runs.append({"N": N, "layers": n_layer, "d": d, "curve": curve, "loss": curve[-1][1]})
    print(f"layers={n_layer} d={d:3d}  N={N:>8,d}  val loss={curve[-1][1]:.3f}   ({time.time() - t0:.0f}s)")

## 3 · Fit a power law in N
Kaplan's form is `L(N) = (N_c / N)^alpha`; with small models it helps to add a constant `c`, the loss you could never get below
(the entropy of the text itself). A power law is a straight line on a log-log plot of `L − c`.

In [ ]:
Ns = np.array([r["N"] for r in runs], dtype=float); Ls = np.array([r["loss"] for r in runs])
power = lambda N, a, alpha, c: a * N ** (-alpha) + c
(a, alpha, c), _ = curve_fit(power, Ns, Ls, p0=[10, 0.3, 1.0], bounds=([0, 0, 0], [1e4, 2, Ls.min()]), maxfev=20000)
print(f"fit: L(N) = {a:.2f} * N^(-{alpha:.3f}) + {c:.3f}")

grid = np.logspace(np.log10(Ns.min() / 2), np.log10(5e6), 100)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(Ns, Ls, "o", label="trained"); ax[0].plot(grid, power(grid, a, alpha, c), "--", label="fit")
ax[0].set_xscale("log"); ax[0].set_xlabel("non-embedding parameters N"); ax[0].set_ylabel("validation loss (nats/char)"); ax[0].legend()
ax[1].loglog(Ns, Ls - c, "o"); ax[1].loglog(grid, a * grid ** (-alpha), "--")
ax[1].set_xlabel("N"); ax[1].set_ylabel("loss − c"); ax[1].set_title("a straight line on log-log = a power law")
plt.tight_layout(); plt.show()
for N_big in [2e6, 5e6]:
    print(f"predicted loss at N = {N_big:,.0f}: {power(N_big, a, alpha, c):.3f}")

## 4 · Loss vs. compute: the compute-efficient frontier
Training compute is roughly `C ≈ 6 · N · D` FLOPs (forward + backward for D tokens). Plot every model's learning curve against compute.
Small models are best for tiny budgets, bigger ones win as compute grows; the lower envelope is the frontier of slide 11 (left plot).

In [ ]:
plt.figure(figsize=(6.5, 4))
for r in runs:
    C = [6 * r["N"] * D for D, _ in r["curve"]]; L = [l for _, l in r["curve"]]
    plt.plot(C, L, label=f"N={r['N']:,}")
plt.xscale("log"); plt.xlabel("training compute C = 6ND (FLOPs)"); plt.ylabel("validation loss")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.title("Each curve is one model size"); plt.show()

## 5 · Chinchilla: how to spend a compute budget (slide 102)
Hoffmann et al. fitted `L(N, D) = E + A / N^alpha + B / D^beta` to over 400 training runs (their "Approach 3"). For a budget C, set
D = C / (6N) and find the N that minimises the loss. We compare two sets of constants for the same formula:

* the values published by Hoffmann et al. (2022): E = 1.69, A = 406.4, B = 410.7, alpha = 0.34, beta = 0.28;
* a re-fit of the same data by Epoch AI ([Besiroglu et al., 2024](https://arxiv.org/abs/2404.10102)): E = 1.8172, A = 482.01, B = 2085.43,
  alpha = 0.3478, beta = 0.3658.

In [ ]:
FITS = {"Hoffmann et al. 2022": (1.69, 406.4, 410.7, 0.34, 0.28),
        "Epoch AI re-fit 2024": (1.8172, 482.01, 2085.43, 0.3478, 0.3658)}

def loss(fit, N, D):
    E, A, B, al, be = FITS[fit]
    return E + A / N ** al + B / D ** be

def optimal(fit, C):
    Ns = np.logspace(7, 13, 4000); Ds = C / (6 * Ns)
    i = np.argmin(loss(fit, Ns, Ds)); return Ns[i], Ds[i]

for fit in FITS:
    print(fit)
    print(f"  {'compute C':>10s} {'best N':>9s} {'best D':>9s} {'tokens/param':>13s}")
    for C in [1e21, 1e22, 1e23, 5.76e23, 1e24, 1e25]:
        N, D = optimal(fit, C)
        print(f"  {C:10.2e} {N / 1e9:8.1f}B {D / 1e9:8.0f}B {D / N:13.1f}")

The two fits disagree. With the published (rounded) constants the optimum drifts from about 50 to over 100 tokens per parameter (Epoch AI
reports about 70 with the unrounded values), which does not match the paper's own other two methods — or Chinchilla itself: 70B
parameters and 1.4T tokens, i.e. 20 tokens per parameter, trained with Gopher's budget of 5.76e23 FLOPs. The re-fit gives the familiar
**~20 tokens per parameter**, the rule of thumb from the lecture, and at 5.76e23 FLOPs it recommends almost exactly Chinchilla's size. Fitting scaling laws is
delicate: small choices in the fit change the advice. We use the re-fit to place real models on the plot. GPT-3 and Gopher spent their
compute on parameters instead of tokens:

In [ ]:
FIT = "Epoch AI re-fit 2024"
models = {"GPT-3": (175e9, 300e9), "Gopher": (280e9, 300e9), "Chinchilla": (70e9, 1.4e12)}
Cs = np.logspace(21, 25, 50); opt_ND = np.array([optimal(FIT, C) for C in Cs])
plt.figure(figsize=(6.5, 4))
plt.loglog(opt_ND[:, 1], opt_ND[:, 0], "k-", label="compute-optimal (" + FIT + ")")
for name, (N, D) in models.items():
    No, Do = optimal(FIT, 6 * N * D)
    plt.loglog(D, N, "*", ms=14, label=f"{name}: loss {loss(FIT, N, D):.3f} (optimal for same C: {loss(FIT, No, Do):.3f})")
plt.xlabel("training tokens D"); plt.ylabel("parameters N"); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()
N, D = models["GPT-3"]; No, Do = optimal(FIT, 6 * N * D)
print(f"GPT-3's compute would be better spent on a {No / 1e9:.0f}B model trained on {Do / 1e12:.1f}T tokens.")

Why do today's models train far *beyond* 20 tokens per parameter (LLaMA-2 7B on 2T tokens, Qwen3 on 36T)? Because Chinchilla optimises
training compute only. A smaller model trained longer costs more to train but is much cheaper to **serve** to millions of users (slide 102).

## Exercises
1. Train one more model, e.g. `(4, 192, 6)` (about 1.8M parameters). Does its loss land near your extrapolation from section 3?
   What would make it miss?
2. Halve `STEPS`. Which models suffer most? You have entered the *data-limited* regime, where the loss depends on D as well as N.
3. Set `FIT = "Hoffmann et al. 2022"` and redraw the plot. Where does Chinchilla itself sit now? Which fit would you trust, and why?
4. How much compute does a 7B model trained on 2T tokens use, and how much lower would the loss be if the same compute were spent
   compute-optimally?

**Next:** [Notebook 3 — base vs. instruction-tuned: prompting a small LLM]({colab(N3)})